In [1]:
from SPARQLWrapper import SPARQLWrapper, TURTLE, JSON, CSV
import subprocess
import time
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import glob
import itertools
import requests
import csv
from utility import *

In [2]:
current_directory = os.getcwd()
BioPAX_Ontology_file_path = os.path.join(current_directory, '../../Data/BioPAX/BioPAXOntology/biopax-level3.owl')
ReactomeBioPAX_file_path = os.path.join(current_directory, '../../Data/BioPAX/ReactomeBioPAX/Homo_sapiens_v96.owl')
tgf_beta_pathway_path = os.path.join(current_directory, '../../Data/BioPAX/ReactomeBioPAX/R-HSA-9006936.xml')
signaling_tgf_beta_complex = os.path.join(current_directory, '../../Data/BioPAX/ReactomeBioPAX/R-HSA-170834.xml')
PathwayAbstractionsFolder = os.path.join(current_directory, '../../Results/PathwayAbstraction')

prefixes = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX bp3: <http://www.biopax.org/release/biopax-level3.owl#>
"""
endpoint = "http://localhost:3030/tgf_beta"

In [3]:
query_is_a_component_of = """
SELECT DISTINCT ?subPathwayID ?pathwayID
WHERE {
    ?pathway rdf:type bp3:Pathway .
    ?pathway bp3:pathwayComponent ?subPathway .
    ?subPathway rdf:type bp3:Pathway .

    ?pathway bp3:xref ?pathwayXref .
    ?pathwayXref rdf:type bp3:UnificationXref .
    ?pathwayXref bp3:db "Reactome" .
    ?pathwayXref bp3:id ?pathwayID .
    
    ?subPathway bp3:xref ?subPathwayXref .
    ?subPathwayXref rdf:type bp3:UnificationXref .
    ?subPathwayXref bp3:db "Reactome" .
    ?subPathwayXref bp3:id ?subPathwayID .
}
"""

query_next_step_pathway = """
SELECT DISTINCT ?pathwayID ?nextPathwayID
WHERE {
    ?pathway rdf:type bp3:Pathway .
    ?nextPathway rdf:type bp3:Pathway .
    
    ?pathway bp3:pathwayOrder ?pathwayStep .
    ?nextPathway bp3:pathwayOrder ?nextStep .
    
    ?pathwayStep bp3:nextStep ?nextStep .
    
    FILTER (?pathway != ?nextPathway)

    ?pathway bp3:xref ?pathwayXref .
    ?pathwayXref rdf:type bp3:UnificationXref .
    ?pathwayXref bp3:db "Reactome" .
    ?pathwayXref bp3:id ?pathwayID .
    
    ?nextPathway bp3:xref ?nextPathwayXref .
    ?nextPathwayXref rdf:type bp3:UnificationXref .
    ?nextPathwayXref bp3:db "Reactome" .
    ?nextPathwayXref bp3:id ?nextPathwayID .
}
"""

endpoint = "http://localhost:3030/tgf_beta"
command = [
    '/home/cbeust/Softwares/JenaFuseki/apache-jena-fuseki-4.9.0/fuseki-server',
    '--file', signaling_tgf_beta_complex,
    '--file', BioPAX_Ontology_file_path,
    '/tgf_beta'
]
process = subprocess.Popen(command)
time.sleep(30)

sparql = SPARQLWrapper(endpoint)

sparql.setQuery(prefixes + query_is_a_component_of)
sparql.setReturnFormat(CSV)
try:
    results = sparql.query().convert()
    output_filename = os.path.join(f"../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_IsAComponentOf.csv")
    with open(output_filename, 'wb') as f:
        f.write(results)
    print(f"Results saved in {output_filename}")
except Exception as e:
    print(f"Error in IsAComponentOf SPARQL query: {e}")
iscomponentof = pd.read_csv(f"../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_IsAComponentOf.csv", sep=",", header=0)
new_col = ["abstraction:IsAComponentOf"]*len(iscomponentof)
iscomponentof.insert(loc=1, column="interaction", value=new_col)
iscomponentof.to_csv(f"../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_IsAComponentOf.csv", sep=",", header=True, index=False)

sparql.setQuery(prefixes + query_next_step_pathway)
sparql.setReturnFormat(CSV)
try:
    results = sparql.query().convert()
    output_filename = os.path.join("../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_NextStepPathway.csv")
    with open(output_filename, 'wb') as f:
        f.write(results)
    print(f"Results saved in {output_filename}")
except Exception as e:
    print(f"Error in NextStepPathway query: {e}")

nextsteppathway = pd.read_csv(f"../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_NextStepPathway.csv", sep=",", header=0)
new_col = ["abstraction:NextStepPathway"]*len(nextsteppathway)
nextsteppathway.insert(loc=1, column="interaction", value=new_col)
nextsteppathway.to_csv(f"../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_NextStepPathway.csv", sep=",", header=True, index=False)

try:
    q1 = pd.read_csv(os.path.join("../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_IsAComponentOf.csv"), header=None)
    q2 = pd.read_csv(os.path.join("../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_NextStepPathway.csv"), header=None)
    q1 = q1.drop(0).reset_index(drop=True)
    q2 = q2.drop(0).reset_index(drop=True)
    concat_df = pd.concat([q1, q2], ignore_index=True)
    output_csv = os.path.join("../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_PathwayAbstraction.tsv")
    concat_df.to_csv(output_csv, sep=',', header=False, index=False)
    print(f"CSV output saved in {output_csv}")
except Exception as e:
    print(f"Error: {e}")

process.kill()
time.sleep(30)

10:49:17 INFO  Server          :: Dataset: in-memory: load file: /home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/01_PathwayAbstraction/../../Data/BioPAX/ReactomeBioPAX/R-HSA-170834.xml
10:49:18 INFO  Server          :: Dataset: in-memory: load file: /home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/01_PathwayAbstraction/../../Data/BioPAX/BioPAXOntology/biopax-level3.owl
10:49:18 INFO  Server          :: Running in read-only mode for /tgf_beta
10:49:18 INFO  Server          :: Apache Jena Fuseki 4.9.0
10:49:18 INFO  Config          :: FUSEKI_HOME=/home/cbeust/Softwares/JenaFuseki/apache-jena-fuseki-4.9.0
10:49:18 INFO  Config          :: FUSEKI_BASE=/home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/01_PathwayAbstraction/run
10:49:18 INFO  Config          :: Shiro file: file:///home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/01_PathwayAbstraction/run/shiro.ini
10:49:18 INFO  Server          :: Database: in-memory, with fi

Results saved in ../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_IsAComponentOf.csv
Results saved in ../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_NextStepPathway.csv
CSV output saved in ../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_PathwayAbstraction.tsv


In [4]:
# query_up_per_pathway = """ 
# SELECT DISTINCT ?pathwayID ?entityID
# WHERE {
#     VALUES ?db { "UniProt" "UniProt Isoform" }
#     ?pathway rdf:type bp3:Pathway .
#     ?pathway (bp3:pathwayComponent)*|(bp3:pathwayOrder/bp3:stepProcess)* ?interaction .
#     ?interaction rdf:type/(rdfs:subClassOf*) bp3:Interaction .
#     ?pathway bp3:xref ?pathwayXref .
#     ?pathwayXref rdf:type bp3:UnificationXref .
#     ?pathwayXref bp3:db "Reactome" .
#     ?pathwayXref bp3:id ?pathwayID .

#     VALUES ?relation { bp3:left bp3:right bp3:participant bp3:controller }
#     ?interaction ?relation ?entity .

#     ?entity (bp3:component*)|(bp3:memberPhysicalEntity*)|(bp3:component*/bp3:memberPhysicalEntity*)|(bp3:memberPhysicalEntity*/bp3:component*)|(bp3:component*/bp3:memberPhysicalEntity*/bp3:component*) ?entityCompo .
#     ?entityCompo rdf:type/(rdfs:subClassOf*) bp3:PhysicalEntity .
#     ?entityCompo bp3:entityReference ?entityRef .
#     ?entityRef bp3:xref ?entityRefXref .
#     ?entityRefXref rdf:type bp3:UnificationXref .
#     ?entityRefXref bp3:db ?db .
#     ?entityRefXref bp3:id ?entityID .
# } 
# """

# query_nb_up_per_pathway = """ 
# SELECT ?pathwayID (COUNT(DISTINCT ?entityID) AS ?nbID)
# WHERE {
#     VALUES ?db { "UniProt" "UniProt Isoform" }
#     ?pathway rdf:type bp3:Pathway .
#     ?pathway bp3:displayName ?pathwayName .
#     ?pathway (bp3:pathwayComponent)*|(bp3:pathwayOrder/bp3:stepProcess)* ?interaction .
#     ?interaction rdf:type/(rdfs:subClassOf*) bp3:Interaction .
#     ?pathway bp3:xref ?pathwayXref .
#     ?pathwayXref rdf:type bp3:UnificationXref .
#     ?pathwayXref bp3:db "Reactome" .
#     ?pathwayXref bp3:id ?pathwayID .

#     VALUES ?relation { bp3:left bp3:right bp3:participant bp3:controller }
#     ?interaction ?relation ?entity .

#     ?entity (bp3:component*)|(bp3:memberPhysicalEntity*)|(bp3:component*/bp3:memberPhysicalEntity*)|(bp3:memberPhysicalEntity*/bp3:component*)|(bp3:component*/bp3:memberPhysicalEntity*/bp3:component*) ?entityCompo .
#     ?entityCompo rdf:type/(rdfs:subClassOf*) bp3:PhysicalEntity .
#     ?entityCompo bp3:entityReference ?entityRef .
#     ?entityRef bp3:xref ?entityRefXref .
#     ?entityRefXref rdf:type bp3:UnificationXref .
#     ?entityRefXref bp3:db ?db .
#     ?entityRefXref bp3:id ?entityID .
# }
# GROUP BY ?pathwayID
# """

query_up_per_pathway = """ 
SELECT DISTINCT ?pathwayID ?entityID
WHERE {
    VALUES ?db { "UniProt" "UniProt Isoform" }

    ?pathway rdf:type bp3:Pathway .
    ?pathway (bp3:pathwayComponent | bp3:pathwayOrder/bp3:stepProcess)* ?interaction .
    ?interaction rdf:type/(rdfs:subClassOf*) bp3:Interaction .
    ?pathway bp3:xref ?pathwayXref .
    ?pathwayXref rdf:type bp3:UnificationXref ;
                    bp3:db "Reactome" ;
                    bp3:id ?pathwayID .

    VALUES ?relation { bp3:left bp3:right bp3:participant bp3:controller }
    ?interaction ?relation ?entity .
    
    ?entity (bp3:component | bp3:memberPhysicalEntity)* ?entityCompo .

    ?entityCompo rdf:type/(rdfs:subClassOf*) bp3:PhysicalEntity .
    ?entityCompo bp3:entityReference ?entityRef .
    ?entityRef bp3:xref ?entityRefXref .
    ?entityRefXref rdf:type bp3:UnificationXref ;
                    bp3:db ?db ;
                    bp3:id ?entityID .
}
"""

query_nb_up_per_pathway = """ 
SELECT ?pathwayID (COUNT(DISTINCT ?entityID) AS ?nbID)
WHERE {
    VALUES ?db { "UniProt" "UniProt Isoform" }

    ?pathway rdf:type bp3:Pathway .
    ?pathway (bp3:pathwayComponent | bp3:pathwayOrder/bp3:stepProcess)* ?interaction .
    ?interaction rdf:type/(rdfs:subClassOf*) bp3:Interaction .
    ?pathway bp3:xref ?pathwayXref .
    ?pathwayXref rdf:type bp3:UnificationXref ;
                    bp3:db "Reactome" ;
                    bp3:id ?pathwayID .

    VALUES ?relation { bp3:left bp3:right bp3:participant bp3:controller }
    ?interaction ?relation ?entity .
    
    ?entity (bp3:component | bp3:memberPhysicalEntity)* ?entityCompo .

    ?entityCompo rdf:type/(rdfs:subClassOf*) bp3:PhysicalEntity .
    ?entityCompo bp3:entityReference ?entityRef .
    ?entityRef bp3:xref ?entityRefXref .
    ?entityRefXref rdf:type bp3:UnificationXref ;
                    bp3:db ?db ;
                    bp3:id ?entityID .
}
GROUP BY ?pathwayID
"""

endpoint = "http://localhost:3030/tgf_beta"

command = [
    '/home/cbeust/Softwares/JenaFuseki/apache-jena-fuseki-4.9.0/fuseki-server',
    '--file', signaling_tgf_beta_complex,
    '--file', BioPAX_Ontology_file_path,
    '/tgf_beta'
]
print("Fuseki command:", command)

process = subprocess.Popen(command)
time.sleep(30)

sparql = SPARQLWrapper(endpoint)

sparql.setQuery(prefixes + query_up_per_pathway)
sparql.setReturnFormat(CSV)
try:
    results = sparql.query().convert()
    output_filename = os.path.join("../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_UpPerPathway.csv")
    with open(output_filename, 'wb') as f:
        f.write(results)
    print(f"Results saved in {output_filename}")
except Exception as e:
    print(f"Error: {e}")

sparql.setQuery(prefixes + query_nb_up_per_pathway)
sparql.setReturnFormat(CSV)
try:
    results = sparql.query().convert()
    output_filename = os.path.join("../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_NbUpPerPathway.csv")
    with open(output_filename, 'wb') as f:
        f.write(results)
    print(f"Results saved in {output_filename}")
except Exception as e:
    print(f"Error: {e}")

process.kill()
time.sleep(30)

Fuseki command: ['/home/cbeust/Softwares/JenaFuseki/apache-jena-fuseki-4.9.0/fuseki-server', '--file', '/home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/01_PathwayAbstraction/../../Data/BioPAX/ReactomeBioPAX/R-HSA-170834.xml', '--file', '/home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/01_PathwayAbstraction/../../Data/BioPAX/BioPAXOntology/biopax-level3.owl', '/tgf_beta']


10:50:58 INFO  Server          :: Dataset: in-memory: load file: /home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/01_PathwayAbstraction/../../Data/BioPAX/ReactomeBioPAX/R-HSA-170834.xml
10:50:59 INFO  Server          :: Dataset: in-memory: load file: /home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/01_PathwayAbstraction/../../Data/BioPAX/BioPAXOntology/biopax-level3.owl
10:50:59 INFO  Server          :: Running in read-only mode for /tgf_beta
10:50:59 INFO  Server          :: Apache Jena Fuseki 4.9.0
10:50:59 INFO  Config          :: FUSEKI_HOME=/home/cbeust/Softwares/JenaFuseki/apache-jena-fuseki-4.9.0
10:50:59 INFO  Config          :: FUSEKI_BASE=/home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/01_PathwayAbstraction/run
10:50:59 INFO  Config          :: Shiro file: file:///home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/01_PathwayAbstraction/run/shiro.ini
10:50:59 INFO  Server          :: Database: in-memory, with fi

Results saved in ../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_UpPerPathway.csv


10:51:29 INFO  Fuseki          :: [2] 200 OK (212 ms)


Results saved in ../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_NbUpPerPathway.csv


In [5]:
query = """
SELECT (COUNT(DISTINCT ?id) AS ?nbID)
WHERE {
VALUES ?db { "UniProt" "UniProt Isoform" }
?entityRef rdf:type/(rdfs:subClassOf*) bp3:ProteinReference .
?entityRef bp3:xref ?entityRefXref .
?entityRefXref bp3:db ?db .
?entityRefXref bp3:id ?id .
}
"""

isacomponentof = pd.read_csv(f"../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_IsAComponentOf.csv", sep=',', header=0)
nextsteppathway = pd.read_csv(f"../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_NextStepPathway.csv", sep=',', header=0)
G = create_networkx_graph(isacomponentof)
root = [node for node in G.nodes() if G.out_degree(node) == 0]

up_per_pathway = pd.read_csv(f"../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_UpPerPathway.csv", sep=",", header=0)

dico_er_per_pathway = dict()
for index, row in up_per_pathway.iterrows():
    pathway = row[0]
    id = row[1]
    if not pathway in dico_er_per_pathway.keys():
        dico_er_per_pathway[pathway] = [id]
    else:
        dico_er_per_pathway[pathway] += [id]

command = [
    '/home/cbeust/Softwares/JenaFuseki/apache-jena-fuseki-4.9.0/fuseki-server',
    '--file', signaling_tgf_beta_complex,
    '--file', BioPAX_Ontology_file_path,
    '/tgf_beta'
]
print("Fuseki command:", command)

process = subprocess.Popen(command)
time.sleep(30)
sparql = SPARQLWrapper(endpoint)
sparql.setQuery(prefixes + query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()
nbEr_total = int(results["results"]["bindings"][0]["nbID"]["value"])
print(nbEr_total)

dico_isacomponentof_resnik_er = dict()
for index, row in isacomponentof.iterrows():
    pathway1 = row[0]
    pathway2 = row[2]
    interaction = row[1]
    if interaction == "abstraction:IsAComponentOf":
        score = resnik_similarity_ER(G, root[0], pathway1, pathway2, dico_er_per_pathway, nbEr_total)
        key = f"{str(pathway1)}-<>{str(pathway2)}"
        dico_isacomponentof_resnik_er[key] = score
print(dico_isacomponentof_resnik_er)

weighted_graph = pd.DataFrame(columns=['source', 'interaction', 'target', 'weight'])
counter_rows = 0
for index, row in isacomponentof.iterrows():
    weighted_graph.at[counter_rows, 'source'] = row[0]
    weighted_graph.at[counter_rows, 'interaction'] = row[1]
    weighted_graph.at[counter_rows, 'target'] = row[2]
    key = f'{str(row[0])}-<>{str(row[2])}'
    weight = dico_isacomponentof_resnik_er[key]
    weighted_graph.at[counter_rows, 'weight'] = weight
    counter_rows += 1

weighted_graph.to_csv(f"../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_IsAComponentOf_ResnikER.csv", sep=",", header=True, index=False)

process.kill()
time.sleep(30)

# WEIGHT NEXTSTEPPATHWAY EDGES
dico_next_step_er = dict()
distribution_weight_er = list()

for item, row in nextsteppathway.iterrows():
    if row[1] == "abstraction:NextStepPathway":
        Pathway1 = row[0]
        Pathway2 = row[2]
        key = f"{str(Pathway1)}->{str(Pathway2)}"
        weight_er = compute_similarity(Pathway1, Pathway2, dico_er_per_pathway, nbEr_total)
        distribution_weight_er.append(weight_er)
        dico_next_step_er[key] = weight_er

weighted_graph = pd.DataFrame(columns=['source', 'interaction', 'target', 'weight'])
counter_rows = 0
for item, row in nextsteppathway.iterrows():
    if row[1] == "abstraction:NextStepPathway":
        weighted_graph.at[counter_rows, 'source'] = row[0]
        weighted_graph.at[counter_rows, 'interaction'] = row[1]
        weighted_graph.at[counter_rows, 'target'] = row[2]
        key = f"{str(row[0])}->{str(row[2])}"
        weight = dico_next_step_er[key]
        weighted_graph.at[counter_rows, 'weight'] = weight
    counter_rows += 1
weighted_graph.to_csv(f"../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_NextStepPathway_ERcontent.csv", sep=",", header=True, index=False)

# CONCATENATE GRAPHS
weighted_is_a_component_of = pd.read_csv(f"../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_IsAComponentOf_ResnikER.csv", sep=",", header=0)
print(weighted_is_a_component_of.head())

weighted_next_step_pathway = pd.read_csv(f"../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_NextStepPathway_ERcontent.csv", sep=",", header=0)
print(weighted_next_step_pathway.head())

global_abstraction = pd.concat([weighted_is_a_component_of, weighted_next_step_pathway], axis=0)
global_abstraction.to_csv(f"../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_WeightedPathwayAbstraction.csv", sep=",", index=False)

Pathway abstraction loaded in networkx: MultiDiGraph with 7 nodes and 6 edges
Fuseki command: ['/home/cbeust/Softwares/JenaFuseki/apache-jena-fuseki-4.9.0/fuseki-server', '--file', '/home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/01_PathwayAbstraction/../../Data/BioPAX/ReactomeBioPAX/R-HSA-170834.xml', '--file', '/home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/01_PathwayAbstraction/../../Data/BioPAX/BioPAXOntology/biopax-level3.owl', '/tgf_beta']


/tmp/ipykernel_59697/3808403658.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pathway = row[0]
/tmp/ipykernel_59697/3808403658.py:22: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  id = row[1]
10:52:46 INFO  Server          :: Dataset: in-memory: load file: /home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/01_PathwayAbstraction/../../Data/BioPAX/ReactomeBioPAX/R-HSA-170834.xml
10:52:47 INFO  Server          :: Dataset: in-memory: load file: /home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/01_PathwayAbstraction/../../Data/BioPAX/BioPAXOntology/biopax-level3.owl
10:52:47 INFO  Ser

93
{'R-HSA-2173791-<>R-HSA-170834': 0, 'R-HSA-2173793-<>R-HSA-170834': 0, 'R-HSA-2173789-<>R-HSA-170834': 0, 'R-HSA-2173795-<>R-HSA-2173793': 0.6007738604289302, 'R-HSA-2173796-<>R-HSA-2173793': 0.6007738604289302, 'R-HSA-2173788-<>R-HSA-2173789': 0.703958096664161}
          source                 interaction         target    weight
0  R-HSA-2173791  abstraction:IsAComponentOf   R-HSA-170834  0.000000
1  R-HSA-2173793  abstraction:IsAComponentOf   R-HSA-170834  0.000000
2  R-HSA-2173789  abstraction:IsAComponentOf   R-HSA-170834  0.000000
3  R-HSA-2173795  abstraction:IsAComponentOf  R-HSA-2173793  0.600774
4  R-HSA-2173796  abstraction:IsAComponentOf  R-HSA-2173793  0.600774
          source                  interaction         target    weight
0  R-HSA-2173795  abstraction:NextStepPathway  R-HSA-2173788  0.682452
1  R-HSA-2173795  abstraction:NextStepPathway  R-HSA-2173796  0.600774
2  R-HSA-2173796  abstraction:NextStepPathway  R-HSA-2173795  0.600774
3  R-HSA-2173789  abstraction

/tmp/ipykernel_59697/3808403658.py:77: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if row[1] == "abstraction:NextStepPathway":
/tmp/ipykernel_59697/3808403658.py:78: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Pathway1 = row[0]
/tmp/ipykernel_59697/3808403658.py:79: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Pathway2 = row[2]
/tmp/ipykernel_59697/3808403658.py:88: FutureWarning: Series.__getitem__ treating keys as positions i

In [6]:
process.kill()
time.sleep(30)